# LCA Lasers4MaaS: joining processes

Unit processes of the joining step for 5 use cases, with State of the Art (SoA) technologies and with Dynamic Beam Shaping (DBS).
The functional unit is 1 joined unit (`1 unit`).

The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [21]:
import os
import bw2data as bd
from IPython.display import display
from l4m_lca import plots
from l4m_lca.constants import BACKGROUNDS, EU27, EUROSTAT_ELEC_PRICE_2025S1
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.joining import joining_processes, technology_of
from l4m_lca.lcia import COST_COLUMN, ELECTRICITY_COLUMN, export, scenario_table
from l4m_lca.project import DEFAULT_INDICATORS, category, ef31_methods, find_activity, method_units, open_project
from l4m_lca.regional import electricity_markets, regional_table

## Parameters

In [22]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BACKGROUND = os.environ.get("L4M_BACKGROUND", "bafu")
PROFILE = BACKGROUNDS[BACKGROUND]
PROJECT_NAME = PROFILE["project"]
BACKGROUND_DATABASE = PROFILE["database"]
RESULTS_DIR = PROFILE["results"]

SOA_NAME = "State_of_the_Art"
DBS_NAME = "Dynamic_Beam_Shaping"

INDICATORS = DEFAULT_INDICATORS            # EF 3.1 indicators (substrings of the method names)

# Background activities of each background database: (name, location)
ELECTRICITY = {"ecoinvent": ("market group for electricity, medium voltage", "RER"),
               "bafu": ("Electricity, medium voltage, at grid", "RER")}
SHIELDING_GAS = {"ecoinvent": ("market for argon, liquid", "RER"),
                 "bafu": ("Argon, liquid, at plant", "RER")}

# Prices of the main resource inputs
COST_FACTORS = {
    "electricity_EUR_per_kWh": 0.29,   # industrial electricity
    "argon_EUR_per_kg": 5.87,          # argon gas
}

# Regional analysis: industrial electricity price of each country (EUR/kWh); argon keeps its price above
ELEC_PRICE = EUROSTAT_ELEC_PRICE_2025S1
REGIONAL_BASELINE = "RER"

In [23]:
# Joining parameters of the SoA processes:
# joining power and welding station power (kW), joining speed (m/s), joint length (m), gas flow (kg/s).
# minimum = lowest power and gas flow at the fastest speed; maximum = highest at the slowest speed.
SOA_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.4, "max": 1.5},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.012, "max": 0.014},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.8, "max": 2.52},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.0025, "max": 0.0025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000268, "max": 0.000268},
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.4, "max": 2.1},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.015, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000892, "max": 0.000892},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.8, "max": 2.0},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.009, "max": 0.009},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.001487, "max": 0.001487},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 0, "max": 0},
        "power_kW_station": {"min": 2000, "max": 2000},
        "joining_speed_m_per_s": {"min": 0.460, "max": 0.460},
        "joint_length_m": 27.5,
        "gas_flow_kg_per_s": {"min": 0.123889, "max": 0.123889},
    },
}

# SoA technology of each use case
SOA_TECHNOLOGY = {
    "Heat exchanger": "standard laser",
    "Fuel tank": "TIG",
    "Heterogeneous pipe": "standard laser",
    "Freeze Dryers Shelf": "standard laser",
    "Battery cooling plate": "controlled atmosphere brazing",
}

In [24]:
# Joining parameters with DBS. power_kW_station includes the chiller.
DBS_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.26, "max": 1.35},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.62, "max": 2.268},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.26, "max": 1.89},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.62, "max": 1.8},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.1, "max": 0.1},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 8, "max": 10},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joint_length_m": 27.5,
        "joining_speed_m_per_s": {"min": 0.4, "max": 0.4},
        "gas_flow_kg_per_s": {"min": 0, "max": 0},
    },
}

## Project and background data

In [25]:
background = open_project(PROJECT_NAME, BACKGROUND_DATABASE)
elec = find_activity(BACKGROUND_DATABASE, *ELECTRICITY[BACKGROUND])
gas = find_activity(BACKGROUND_DATABASE, *SHIELDING_GAS[BACKGROUND])
print(f"Background: {BACKGROUND_DATABASE} (project {PROJECT_NAME}), results in {RESULTS_DIR}/")

methods = ef31_methods(INDICATORS)
units = method_units(methods)
for m in methods:
    print(f"  {m}  ->  unit: {units[category(m)]}")

impacts = BackgroundImpacts(methods, elec)
prices = {elec.key: COST_FACTORS["electricity_EUR_per_kWh"], gas.key: COST_FACTORS["argon_EUR_per_kg"]}

Background: BAFU-2026 (project Lasers4MaaS_BAFU), results in results/bafu/
  ('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')  ->  unit: mol H+-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'eutrophication: terrestrial', 'accumulated exceedance (AE)')  ->  unit: mol N-Eq
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, inorganics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, organics', 'comparative toxic unit for human (CTUh)')  ->  unit: CTUh
  ('ecoinvent-3.12', 'EF v3.1', 'particulate matter formation', 'impact on human health')  ->  unit: disease incidence
  ('ecoinvent-3.12', 'EF v3.1', 'photochemical oxidant formation: human health', 'tropospheric ozone concentration increase')  ->  unit: kg NMVOC-Eq
  ('ecoinvent-3.12', 'EN15804+A2 - Core impact cate

## Unit processes

Electricity and argon exchanges come from the parameters above. Each exchange has an amount (the midpoint) and a minimum and maximum for the scenarios.

In [26]:
soa = bd.Database(SOA_NAME)
soa.write(joining_processes(SOA_NAME, SOA_JOINING_PARAMS, SOA_TECHNOLOGY, elec, gas))

dbs = bd.Database(DBS_NAME)
dbs.write(joining_processes(DBS_NAME, DBS_JOINING_PARAMS, "Dynamic Beam Shaping", elec, gas))

100%|██████████| 5/5 [00:00<00:00, 84562.58it/s]

16:02:49+0100 [info     ] Vacuuming database            



100%|██████████| 5/5 [00:00<00:00, 81920.00it/s]

16:02:59+0100 [info     ] Vacuuming database            


## LCIA results

Impacts, foreground electricity and consumable costs of each process in the minimum, amount and maximum scenarios.

In [27]:
lcia_df = scenario_table([soa, dbs], impacts, prices)
export(lcia_df, units, RESULTS_DIR, f"{SOA_NAME}_{DBS_NAME}")
lcia_df

Results saved to results/bafu/State_of_the_Art_Dynamic_Beam_Shaping_LCIA_results.csv and results/bafu/State_of_the_Art_Dynamic_Beam_Shaping_LCIA_results.json


,Database,Process,Key,Scenario,acidification,eutrophication: terrestrial,human toxicity: carcinogenic,"human toxicity: carcinogenic, inorganics","human toxicity: carcinogenic, organics",particulate matter formation,photochemical oxidant formation: human health,climate change: total (EF v3.1 - IPCC 2021),Electricity Consumption (kWh),Consumable costs (EUR)
0,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,0.041943,0.078195,8.381289e-09,1.546513e-09,6.834776e-09,2.944528e-07,0.025723,11.687846,33.212560,53.107255
1,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,amount,0.041943,0.078195,8.381289e-09,1.546513e-09,6.834776e-09,2.944528e-07,0.025723,11.687846,33.212560,53.107255
2,State_of_the_Art.bbb12d81,Battery cooling plate joined with controlled a...,Battery cooling plate,maximum,0.041943,0.078195,8.381289e-09,1.546513e-09,6.834776e-09,2.944528e-07,0.025723,11.687846,33.212560,53.107255
3,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,minimum,0.012116,0.022580,2.422528e-09,4.468778e-10,1.975650e-09,8.502989e-08,0.007429,3.376721,9.574074,15.578560
4,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,amount,0.012162,0.022668,2.431526e-09,4.485632e-10,1.982963e-09,8.536160e-08,0.007458,3.389573,9.614815,15.590375
5,State_of_the_Art.bbb12d81,Freeze Dryers Shelf joined with standard laser,Freeze Dryers Shelf,maximum,0.012209,0.022756,2.440525e-09,4.502486e-10,1.990276e-09,8.569332e-08,0.007487,3.402425,9.655556,15.602190
6,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,minimum,0.015369,0.029070,2.994789e-09,5.592127e-10,2.435576e-09,1.093360e-07,0.009498,4.256582,13.212222,7.015620
7,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,amount,0.015600,0.029508,3.039493e-09,5.675858e-10,2.471907e-09,1.109840e-07,0.009641,4.320431,13.414622,7.074316
8,State_of_the_Art.bbb12d81,Fuel tank joined with TIG,Fuel tank,maximum,0.015831,0.029946,3.084198e-09,5.759589e-10,2.508239e-09,1.126320e-07,0.009784,4.384280,13.617022,7.133012
9,State_of_the_Art.bbb12d81,Heat exchanger joined with standard laser,Heat exchanger,minimum,0.026919,0.049526,5.500490e-09,1.004450e-09,4.496040e-09,1.867010e-07,0.016396,7.543078,19.662500,53.836293


In [28]:
all_units = {**units, ELECTRICITY_COLUMN: "kWh", COST_COLUMN: "EUR"}
indicator_cols = [c for c in lcia_df.columns if c not in ("Database", "Process", "Key", "Scenario")]
display(plots.scenario_bars(lcia_df, list(SOA_JOINING_PARAMS), [SOA_NAME, DBS_NAME], all_units,
                            technology_of, indicator_cols))

## Regional analysis: EU electricity markets

Replaces the RER electricity market with the market of each member state. The map shows the impacts and consumable costs per joined unit.
The file `results/regional_joining.csv` is an input of `workflow_assembly.ipynb`.

In [29]:
markets = electricity_markets(background, EU27, PROFILE["country_electricity"])
regional_df = regional_table([soa, dbs], impacts, markets, ELEC_PRICE, prices, REGIONAL_BASELINE)
regional_df["technology"] = regional_df["process"].map(technology_of)
regional_df.to_csv(f"{RESULTS_DIR}/regional_joining.csv", index=False)
regional_df.head()

,database,process,key,scenario,country,indicator,impact,technology
0,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,acidification,4.194290e-02,controlled atmosphere brazing
1,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,eutrophication: terrestrial,7.819529e-02,controlled atmosphere brazing
2,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,human toxicity: carcinogenic,8.381289e-09,controlled atmosphere brazing
3,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,"human toxicity: carcinogenic, inorganics",1.546513e-09,controlled atmosphere brazing
4,State_of_the_Art,Battery cooling plate joined with controlled a...,Battery cooling plate,minimum,RER,"human toxicity: carcinogenic, organics",6.834776e-09,controlled atmosphere brazing


In [30]:
amount_df = regional_df[regional_df["scenario"] == "amount"]
display(plots.regional_explorer(
    amount_df,
    filters=[("Use case", "key", list(SOA_JOINING_PARAMS), None),
             ("Technology", "database", [SOA_NAME, DBS_NAME], None)],
    units=all_units,
    title=lambda f: f"{f['key']}, {f['database'].replace('_', ' ')}: per joined unit",
    baseline=REGIONAL_BASELINE,
))